# Robot Firmware Simulation Demo

Drives the Nezha firmware through **`SimConnection`** — a drop-in replacement
for `SerialConnection` that loads `libfirmware_host.dylib` (MockHAL + Robot +
CommandProcessor) via ctypes.  No hardware required.

| # | Demo | Commands |
|---|------|----------|
| 1 | Square trajectory | `T` + `wait_for_evt_done` |
| 2 | Distance-controlled drive | `D` |
| 3 | Motor asymmetry / drift | `set_motor_offset` |
| 4 | VW velocity ramp + arc | `VW`, `+`, `X` |
| 5 | HALT stop condition | `HALT TIME` |


## 1  Setup

In [ ]:
%matplotlib inline

In [ ]:
import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")
# Force non-interactive backend before matplotlib is imported.
# The default 'macosx' backend deadlocks in a Jupyter kernel (non-main thread).

import ctypes, math, subprocess, pathlib, sys
import matplotlib.pyplot as plt
import numpy as np

# Locate repo root.  Notebooks don't have __file__ so we use cwd.
# Assumes the notebook is run from host_tests/ or the repo root.
CWD = pathlib.Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "host_tests" else CWD
HOST_DIR  = REPO_ROOT / "host"
if str(HOST_DIR) not in sys.path:
    sys.path.insert(0, str(HOST_DIR))

# Build the shared library (output streamed so cell doesn't look frozen).
LIB_DIR = REPO_ROOT / "host_tests" / "build"
if not LIB_DIR.exists():
    print("Configuring CMake...")
    subprocess.run(
        ["cmake", "-S", str(REPO_ROOT / "host_tests"), "-B", str(LIB_DIR)],
        cwd=REPO_ROOT, check=True
    )

print("Building libfirmware_host... (first build ~30 s)")
sys.stdout.flush()
proc = subprocess.Popen(
    ["cmake", "--build", str(LIB_DIR), "--", "-j4"],
    cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Build failed (exit {proc.returncode})")
print("Build OK")

from robot_radio.io.sim_conn import SimConnection
from robot_radio.robot.protocol import NezhaProtocol

print(f"matplotlib backend: {plt.get_backend()}")
print("Imports OK")

In [ ]:
# Helper: create a fresh SimConnection + NezhaProtocol with safety watchdog disabled.
def make_sim(lib_path=None):
    conn  = SimConnection(lib_path=lib_path)
    result = conn.connect()
    if "error" in result:
        raise RuntimeError(result["error"])
    proto = NezhaProtocol(conn)
    proto.set_config(sTimeout=60000)   # disable 500 ms watchdog for demos
    return conn, proto

# Quick smoke test.
conn_test, proto_test = make_sim()
t_robot, rtt = proto_test.ping()
print(f"PING OK  robot_t={t_robot} ms")
conn_test.disconnect()

## 2  Square Trajectory

Drive four 500 mm straight legs with 90° left turns between them.
Each leg:
- **Straight**: `T 200 200 2600` — timed at ±200 mm/s for 2600 ms ≈ 500 mm
- **Turn**:     `T -200 200 560` — spin at ±200 mm/s for 560 ms ≈ 90°
  (trackwidth = 126 mm → arc each side = π/2 × 63 ≈ 99 mm → 495 ms + ramp margin)

Commands go through `NezhaProtocol.timed()` → `wait_for_evt_done("T")`.  
Each `wait_for_evt_done` call drives 100 ms ticks of the sim via
`conn.read_lines(100)`, recording state at every 24 ms step.

In [ ]:
conn, proto = make_sim()

STRAIGHT_L = 200;  STRAIGHT_R = 200;  STRAIGHT_MS = 2600
TURN_L     = -200; TURN_R     = 200;  TURN_MS     = 560
SETTLE_MS  = 150

conn.clear_state_log()
conn.tick(200)           # settle
conn.clear_state_log()   # start log after settle

segments = []  # (label, start_t_ms) for shading the velocity plot

def now_ms():
    return conn.state_log[-1]["time_ms"] if conn.state_log else 0.0

for side in range(4):
    # --- Straight leg ---
    segments.append((f"Leg {side+1}", now_ms()))
    proto.timed(STRAIGHT_L, STRAIGHT_R, STRAIGHT_MS)
    outcome = proto.wait_for_evt_done("T", timeout_ms=STRAIGHT_MS + 1000)
    enc_l = conn.get_state()["enc_l"]
    print(f"Leg {side+1}  straight → {outcome:8s}  enc_l={enc_l:.0f} mm")
    conn.tick(SETTLE_MS)

    # --- Turn ---
    segments.append((f"Turn {side+1}", now_ms()))
    proto.timed(TURN_L, TURN_R, TURN_MS)
    outcome = proto.wait_for_evt_done("T", timeout_ms=TURN_MS + 500)
    h_deg = math.degrees(conn.get_state()["pose_h"])
    print(f"Leg {side+1}  turn     → {outcome:8s}  heading={h_deg:+.1f}°")
    conn.tick(SETTLE_MS)

proto.cancel()
conn.tick(200)

print(f"\nTotal sim time : {now_ms()/1000:.1f} s")
print(f"State log size : {len(conn.state_log)} entries")

### 2a  XY Trajectory

In [ ]:
import pandas as pd

df = conn.state_df()
seg_end_times = [s[1] for s in segments[1:]] + [df["time_ms"].max()]
palette = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(6, 6))

for i, (label, t0) in enumerate(segments):
    t1   = seg_end_times[i]
    mask = (df["time_ms"] >= t0) & (df["time_ms"] <= t1)
    seg  = df[mask]
    ax.plot(seg["pose_x"], seg["pose_y"],
            color=palette[i % len(palette)], linewidth=2, label=label)

ax.plot(df["pose_x"].iloc[0],  df["pose_y"].iloc[0],  "go", markersize=10, label="Start")
ax.plot(df["pose_x"].iloc[-1], df["pose_y"].iloc[-1], "rs", markersize=8,  label="End")

ax.set_xlabel("X (mm)"); ax.set_ylabel("Y (mm)")
ax.set_title("Square Trajectory — Dead-Reckoning Odometry")
ax.set_aspect("equal")
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

### 2b  Velocity Profiles

Blue shading = straight legs. Red shading = turns.  
Straight: both wheels ramp to ≈200 mm/s.  
Turns: left wheel reverses, right stays positive.

In [ ]:
t_s = df["time_ms"] / 1000.0

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_s, df["vel_l"], color="#1f77b4", linewidth=1.5, label="Left wheel")
ax.plot(t_s, df["vel_r"], color="#ff7f0e", linewidth=1.5, label="Right wheel")

for i, (label, t0) in enumerate(segments):
    t1     = seg_end_times[i]
    is_turn = label.startswith("Turn")
    ax.axvspan(t0 / 1000, t1 / 1000, alpha=0.08,
               color="red" if is_turn else "steelblue")
    ax.text((t0 + t1) / 2 / 1000, 215, label,
            ha="center", va="center", fontsize=7.5, color="#333")

ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Velocity (mm/s)")
ax.set_title("Motor Velocity Profiles — Square Drive")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2c  Encoder Accumulation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_s, df["enc_l"], color="#1f77b4", linewidth=1.5, label="enc_l (mm)")
ax.plot(t_s, df["enc_r"], color="#ff7f0e", linewidth=1.5, label="enc_r (mm)")
for i, (label, t0) in enumerate(segments):
    t1 = seg_end_times[i]
    ax.axvspan(t0 / 1000, t1 / 1000, alpha=0.07,
               color="red" if label.startswith("Turn") else "steelblue")
ax.set_xlabel("Time (s)"); ax.set_ylabel("Encoder (mm)")
ax.set_title("Encoder Accumulation — Square Drive")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

conn.disconnect()

## 3  Distance-Controlled Drive (`D` command)

`D <l> <r> <mm>` stops after the average of L+R encoder deltas exceeds `mm`.
The firmware emits `EVT done D` when the target is reached.

In [ ]:
conn_d, proto_d = make_sim()
TARGET_MM = 300

conn_d.clear_state_log()
proto_d.distance(200, 200, TARGET_MM)
outcome = proto_d.wait_for_evt_done("D", timeout_ms=6000)

state = conn_d.get_state()
print(f"D outcome : {outcome}")
print(f"enc_l     : {state['enc_l']:.1f} mm  (target {TARGET_MM})")
print(f"enc_r     : {state['enc_r']:.1f} mm")

df_d = conn_d.state_df()
t_d  = df_d["time_ms"] / 1000

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_d, df_d["enc_l"], label="enc_l", color="#1f77b4")
axes[0].plot(t_d, df_d["enc_r"], label="enc_r", color="#ff7f0e")
axes[0].axhline(TARGET_MM, color="green", linestyle="--", label=f"target {TARGET_MM} mm")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Encoder (mm)")
axes[0].set_title("D command — encoder convergence")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t_d, df_d["vel_l"], label="vel_l", color="#1f77b4")
axes[1].plot(t_d, df_d["vel_r"], label="vel_r", color="#ff7f0e")
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Velocity (mm/s)")
axes[1].set_title("D command — velocity profile")
axes[1].legend(); axes[1].grid(alpha=0.3)

conn_d.disconnect()
plt.tight_layout(); plt.show()

## 4  Motor Asymmetry — Observing Drift

`conn.set_motor_offset(side=1, factor=0.8)` scales the right wheel to
80% of nominal — simulating a worn motor or calibration error.
Drive straight and watch the robot curve left in the XY plot.

In [ ]:
conn_a, proto_a = make_sim()

# Right wheel at 80% efficiency.
conn_a.set_motor_offset(side=1, factor=0.8)

conn_a.clear_state_log()
proto_a.timed(200, 200, 3000)
proto_a.wait_for_evt_done("T", timeout_ms=4500)

df_a = conn_a.state_df()
t_a  = df_a["time_ms"] / 1000

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(df_a["pose_x"], df_a["pose_y"], color="purple", linewidth=2)
axes[0].plot(df_a["pose_x"].iloc[0],  df_a["pose_y"].iloc[0],  "go", markersize=10)
axes[0].plot(df_a["pose_x"].iloc[-1], df_a["pose_y"].iloc[-1], "rs", markersize=8)
axes[0].set_aspect("equal")
axes[0].set_xlabel("X (mm)"); axes[0].set_ylabel("Y (mm)")
axes[0].set_title("Drift: right wheel at 80% efficiency")
axes[0].grid(True, alpha=0.4)

axes[1].plot(t_a, df_a["vel_l"], label="vel_l (100%)", color="#1f77b4")
axes[1].plot(t_a, df_a["vel_r"], label="vel_r (80%)",  color="#ff7f0e")
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Velocity (mm/s)")
axes[1].set_title("Velocity with asymmetric motors")
axes[1].legend(); axes[1].grid(alpha=0.3)

conn_a.disconnect()
plt.tight_layout(); plt.show()

## 5  VW Velocity Ramp — Body-Twist Control

`VW <v_mms> <omega_mrads>` sets body-frame forward speed and yaw rate.
The `+` keepalive resets the watchdog so VW keeps running between sends.

Phase 1: Ramp `v` 0→300→0 mm/s straight (omega=0).  
Phase 2: Constant `v=200` mm/s + `omega=600` mrad/s left arc.

In [ ]:
conn_v, proto_v = make_sim()
conn_v.clear_state_log()

# Phase 1: ramp v 0 → 300 → 0 mm/s, omega=0.
speeds_v = list(range(0, 310, 30)) + list(range(300, -10, -30))
t_phase2_start = None
for v in speeds_v:
    conn_v.send(f"VW {v} 0")
    conn_v.send("+")        # keepalive
    conn_v.tick(150)

conn_v.send("X")
conn_v.tick(300)
t_phase2_start = conn_v.state_log[-1]["time_ms"]

# Phase 2: constant forward + left arc.
for _ in range(30):
    conn_v.send("VW 200 600")
    conn_v.send("+")
    conn_v.tick(100)

conn_v.send("X")
conn_v.tick(200)

df_v = conn_v.state_df()
t_v  = df_v["time_ms"] / 1000

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Velocity profile with phase shading.
axes[0].plot(t_v, df_v["vel_l"], label="vel_l", color="#1f77b4")
axes[0].plot(t_v, df_v["vel_r"], label="vel_r", color="#ff7f0e")
axes[0].axvline(t_phase2_start / 1000, color="grey", linestyle="--", linewidth=1)
axes[0].text(t_phase2_start / 1000 + 0.05, 280, "Phase 2 →", fontsize=8, color="grey")
axes[0].axhline(0, color="black", linewidth=0.6, linestyle="--")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Velocity (mm/s)")
axes[0].set_title("VW: velocity ramp (phase 1) + left arc (phase 2)")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Trajectory.
axes[1].plot(df_v["pose_x"], df_v["pose_y"], color="teal", linewidth=2)
axes[1].plot(df_v["pose_x"].iloc[0],  df_v["pose_y"].iloc[0],  "go", markersize=10, label="Start")
axes[1].plot(df_v["pose_x"].iloc[-1], df_v["pose_y"].iloc[-1], "rs", markersize=8,  label="End")
axes[1].set_aspect("equal")
axes[1].set_xlabel("X (mm)"); axes[1].set_ylabel("Y (mm)")
axes[1].set_title("VW trajectory: straight ramp → left arc")
axes[1].legend(); axes[1].grid(True, alpha=0.4)

conn_v.disconnect()
plt.tight_layout(); plt.show()

## 6  HALT Stop Condition

`HALT TIME <ms>` registers a named stop condition that fires after `ms`
milliseconds of motion, emitting `EVT halt id=<n>` and stopping motors.
Here a 3-second timed drive is cut short by `HALT TIME 800`.

In [ ]:
conn_h, proto_h = make_sim()

halt_resp = conn_h.send("HALT TIME 800")
print("HALT register :", halt_resp["responses"])

conn_h.clear_state_log()

# Start a 3 s drive.  HALT fires at ~800 ms.
proto_h.timed(200, 200, 3000)

# read_lines(2000) ticks 2 s of sim time, stopping when any EVT appears.
lines = conn_h.read_lines(2000, stop_token="EVT")
print("EVT received  :", lines)

conn_h.tick(300)

df_h = conn_h.state_df()
t_h  = df_h["time_ms"] / 1000

# Find approximate halt time (first tick where velocity ~0 after motion).
moving = df_h[df_h["vel_l"].abs() > 10]
halt_t = moving["time_ms"].max() / 1000 if not moving.empty else None

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t_h, df_h["vel_l"], label="vel_l", color="#1f77b4")
axes[0].plot(t_h, df_h["vel_r"], label="vel_r", color="#ff7f0e")
if halt_t:
    axes[0].axvline(halt_t, color="red", linestyle="--",
                    label=f"HALT at {halt_t:.2f} s")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Velocity (mm/s)")
axes[0].set_title("HALT TIME 800 ms cuts 3 s drive short")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df_h["pose_x"], df_h["pose_y"], color="darkorange", linewidth=2)
axes[1].plot(df_h["pose_x"].iloc[0],  df_h["pose_y"].iloc[0],  "go", markersize=10)
axes[1].plot(df_h["pose_x"].iloc[-1], df_h["pose_y"].iloc[-1], "rs", markersize=8)
axes[1].set_aspect("equal")
axes[1].set_xlabel("X (mm)"); axes[1].set_ylabel("Y (mm)")
axes[1].set_title("Trajectory stops at HALT condition")
axes[1].grid(True, alpha=0.4)

conn_h.disconnect()
plt.tight_layout(); plt.show()

## Summary

| Demo | Key result |
|------|------------|
| Square (T) | XY square + 4-segment velocity profile with ramp and spin phases |
| Distance (D) | Stops exactly at encoder target; velocity decelerates to 0 |
| Asymmetry | Right wheel 80% → curved XY, split velocity lines |
| VW ramp | Smooth trapezoidal ramp; left arc shows differential velocity |
| HALT TIME | Drive terminates early; EVT halt fires at ~800 ms |

All demos run through `NezhaProtocol(SimConnection())` — the same API
that talks to real hardware.  Swap in `SerialConnection` to run on the
physical robot without changing any demo code.

```python
# Real hardware:
from robot_radio.io.serial_conn import SerialConnection
conn = SerialConnection()          # <- only change
conn.connect()
proto = NezhaProtocol(conn)
# ... same proto.timed / wait_for_evt_done calls work unchanged
```